# Test with a real data

In [1]:
import xarray as xr
import glob

from pcmdi_metrics.effective_resolution import compute_effective_resolution
import matplotlib.pyplot as plt
import numpy as np

## Prepare input

In [ ]:
#datadir = "/global/cfs/cdirs/m4581/lee1043/DATA/CMIP6/HighResMIP/ECMWF-IFS-LR/hist-1950/6hrPlevPt"
datadir = "/global/cfs/cdirs/m4581/lee1043/DATA/CMIP6/HighResMIP/ECMWF-IFS-LR/highresSST-present/6hrPlevPt"
#datadir = "/home/lee1043/work/DATA/HighResMIP/ECMWF-IFS-LR/hist-1950/6hrPlevPt"
#datadir = "/home/lee1043/work/DATA/HighResMIP/ECMWF-IFS-LR/highresSST-present/6hrPlevPt"
#datadir = "/Users/lee1043/Documents/Research/DATA/HighResMIP/ECMWF-IFS-LR/highresSST-present/6hrPlevPt"

In [3]:
data_list = sorted(glob.glob(f"{datadir}/*/*ECMWF-IFS-LR*_2014*.nc"))
data_list

[]

In [4]:
target_months = ('201403', '201406', '201409', '201412')

data_list = [
    data_path
    for data_path in data_list
    if any(month in data_path for month in target_months)
]

In [5]:
[filepath.split('/')[-1] for filepath in data_list]

[]

In [6]:
ds = xr.open_mfdataset(data_list)

OSError: no files to open

In [ ]:
ds

## Metrics calculation

Takes ~2 min

In [ ]:
%%time

metrics, diagnostics = compute_effective_resolution(
    ds,
    uvar="ua",
    vvar="va",
    levels=(250.0, 500.0),
    model="ECMWF-IFS-LR",
    member="r1i1p1f1",
    save_interim_netcdf=False,
)

result = metrics["ECMWF-IFS-LR"]["r1i1p1f1"]
result

Save interim fime for cross validation (takes longer time ~8 min)

In [ ]:
"""
%%time

metrics, diagnostics = compute_effective_resolution(
    ds,
    uvar="ua",
    vvar="va",
    levels=(250.0, 500.0),
    model="ECMWF-IFS-LR",
    member="r1i1p1f1",
    save_interim_netcdf=True,
)

result = metrics["ECMWF-IFS-LR"]["r1i1p1f1"]
result
"""

## Analysis (temporary interim data)

In [ ]:
#ds_interim = xr.open_dataset("./output/ke_spectra_interim_ECMWF-IFS-LR_r1i1p1f1.nc")
ds_interim = xr.open_dataset("./output/vorticity_divergence_ECMWF-IFS-LR_r1i1p1f1.nc")

ds_interim

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(10, 6),
    constrained_layout=True
)

plots = [
    (ds_interim.vorticity_250.isel(time=0), "Vorticity 250"),
    (ds_interim.divergence_250.isel(time=0), "Divergence 250"),
    (ds_interim.vorticity_500.isel(time=0), "Vorticity 500"),
    (ds_interim.divergence_500.isel(time=0), "Divergence 500"),
]

vmin = min(data.min().item() for data, _ in plots)
vmax = max(data.max().item() for data, _ in plots)

for ax, (data, title) in zip(axes.flat, plots):
    data.plot(ax=ax, vmin=vmin, vmax=vmax, cmap="RdBu_r", add_colorbar=True)
    ax.set_title(title)

plt.show()

In [ ]:
print("Vorticity 250 hPa statistics:")
print(f"  Min: {ds_interim.vorticity_250.min().values:.2e} s^-1")
print(f"  Max: {ds_interim.vorticity_250.max().values:.2e} s^-1")
print(f"  Std: {ds_interim.vorticity_250.std().values:.2e} s^-1")

print("\nExpected: O(10^-4 to 10^-5) s^-1 for 250 hPa winds")

## Analysis (diagnostics)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the kinetic energy spectra
spectra = diagnostics["spectra"]
colors = {"div": "C0", "rot_250": "C1", "rot_500": "C2"}
labels = {"div": "Divergent 250 hPa", "rot_250": "Rotational 250 hPa", 
          "rot_500": "Rotational 500 hPa"}

# Plot divergent at 250 hPa
spec_250 = spectra[250.0]
wavenumber = spec_250["wavenumber"].values
ke_div = spec_250["ke_div"].values
ax.loglog(wavenumber, ke_div, color=colors["div"], linewidth=2, label=labels["div"])

# Plot rotational at 250 hPa
ke_rot_250 = spec_250["ke_rot"].values
ax.loglog(wavenumber, ke_rot_250, color=colors["rot_250"], linewidth=2, label=labels["rot_250"])

# Plot rotational at 500 hPa
spec_500 = spectra[500.0]
ke_rot_500 = spec_500["ke_rot"].values
ax.loglog(wavenumber, ke_rot_500, color=colors["rot_500"], linewidth=2, label=labels["rot_500"])

# Add reference lines for k^-3 and k^-5/3
# Control where reference lines start by specifying y-value at a reference wavenumber
k_ref_start = 10.0       # Reference wavenumber where we set the y-value
y_ref_value = 1e0        # Y-axis value at k_ref_start for both reference lines

k_ref = np.logspace(0, 3, 100)  # wavenumbers from 1 to 1000

# Calculate amplitude to make reference line pass through (k_ref_start, y_ref_value)
# For k^-3: y = A * k^-3, so A = y * k^3
amplitude_k3 = y_ref_value * k_ref_start**3
amplitude_k53 = y_ref_value * k_ref_start**(5/3)

ke_ref_k3 = amplitude_k3 * k_ref**(-3)
ke_ref_k53 = amplitude_k53 * k_ref**(-5/3)

ax.loglog(k_ref, ke_ref_k3, 'k-', linewidth=1.5, alpha=0.6, label=r'$k^{-3}$')
ax.loglog(k_ref, ke_ref_k53, 'k--', linewidth=1.5, alpha=0.6, label=r'$k^{-5/3}$')

ax.set_xlim(10, 240)
ax.set_ylim(1e-3, 1e2)
ax.set_xlabel("Total horizontal wavenumber", fontsize=11)
ax.set_ylabel(r"Kinetic energy per unit mass (m$^2$ s$^{-2}$)", fontsize=11)
ax.legend(loc="upper right", frameon=True, framealpha=0.9)
ax.grid(True, which="both", alpha=0.3, linestyle=":")
ax.set_title("Kinetic energy spectra — ECMWF-IFS-LR", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
metrics

### Compensated vs. Uncompensated Spectra

The `plot_spectra_and_slope` function supports both:

1. **Compensated spectra** (default, `compensate=3.0`): Reproduces Figure 1 from Klaver et al. (2020)
   - Y-axis shows $l^3 E_l$, making a $k^{-3}$ spectrum appear flat
   - Highlights deviations from the reference power laws
   - Easier to see spectral steepening

2. **Uncompensated spectra** (`compensate=None` or `compensate=0`): Reproduces Figure S3 from supplementary material
   - Y-axis shows raw spectrum in m²/s²
   - Power-law slopes appear as straight lines in log-log space
   - Shows the full dynamic range of the spectra

Both styles show the same data but emphasize different aspects of the spectral behavior.

In [ ]:
from pcmdi_metrics.effective_resolution.lib import (
    plot_spectra_and_slope,
)

In [ ]:
# Test uncompensated spectra (Figure S3 style)
# Set compensate=None or compensate=0 to plot without l^-3 compensation
fig = plot_spectra_and_slope(
    diagnostics, 
    result,
    #compensate=None,         # Turn OFF compensation (Figure S3)
    xlim=(10, 240),          # Wavenumber range
    #spec_ylim=(1e-2, 1e2),   # Uncompensated spectrum y-range
    slope_ylim=(1, 5),       # Slope panel y-range
    title="ECMWF-IFS-LR (compensated spectra)"
)
plt.show()

In [ ]:
# Test uncompensated spectra (Figure S3 style)
# Set compensate=None or compensate=0 to plot without l^-3 compensation
fig = plot_spectra_and_slope(
    diagnostics, 
    result,
    compensate=None,         # Turn OFF compensation (Figure S3)
    xlim=(10, 240),          # Wavenumber range
    spec_ylim=(1e-3, 1e2),   # Uncompensated spectrum y-range
    slope_ylim=(1, 5),       # Slope panel y-range
    title="ECMWF-IFS-LR (uncompensated spectra)"
)
plt.show()

## Figure 2 — Effective vs. Nominal Resolution Scatter Plot

This reproduces Figure 2 from Klaver et al. (2020), showing the relationship between effective resolution (L_eff) and grid box distance (L_box). The ratio L_eff/L_box is color-coded and ranges from 2.7 to 4.8 across the HighResMIP models in the paper.

**Note**: This example shows only one configuration (ECMWF-IFS-LR regridded to 1°). The scatter plot is most useful when comparing multiple models or resolution configurations of the same model (e.g., LR vs HR versions).

In [ ]:
from pcmdi_metrics.effective_resolution.lib import plot_resolution_scatter

# For this example, we only have one model configuration
# In practice, you would have multiple models or configurations to compare
ensemble = [result]
labels = ["ECMWF-IFS-LR (regridded 1°)"]

# Note: The plot shows this is regridded data (not native grid)
# The effective resolution couldn't be determined (None) because only 1 spectrum steepened
# For proper analysis, use native grid data (gn) instead of regridded (gr)

fig = plot_resolution_scatter(
    ensemble, 
    labels=labels, 
    title="Effective vs. Nominal Resolution\n(ECMWF-IFS-LR regridded data)"
)
plt.show()

print("\nNote: This data point shows:")
print(f"  - Grid box distance: {result['grid_box_distance_km']:.1f} km (1° regridded grid)")
print(f"  - Effective resolution: {result['effective_resolution_km']} (if None: insufficient steepening)")
print(f"  - Only 1 of 3 spectra steepened (need 2+ for confirmation)")
print("\nFrom Klaver et al. Table 1, native ECMWF-IFS-LR should have:")
print("  - Grid box distance: 79.6 km (TCO199 reduced Gaussian)")
print("  - Effective wavenumber: l=79")
print("  - Effective resolution: 253 km")
print("  - Ratio: 3.2")